# 03 · Baseline and due-to decomposition

Phase 3 results. Following the RGMx top-down approach, each product-week's **baseline**, the volume it would sell at its regular price without promotion, is

`baseline = exp(ln units − price effect − promo effect − pantry effect − promo-week error)`

- **price effect** = promo elasticity × ln(price / base), the temporary-discount deviation only;
- **promo effect** = the feature, display and TPR contribution from the product's selected model, in promo weeks;
- **pantry effect** = the pl1–pl3 contribution, which restores the post-promotion dip;
- **error** = the positive residual in promo weeks.

Holidays, seasonality and trend stay in the baseline. Implausible promo-week baselines are repaired with RGMx's bad-baseline rules, using a 7-week centred rolling mean. Logic lives in `src/baseline/`; run `make baseline` first.

In [ ]:
# reload edited src/ modules automatically (avoids stale imports after code changes)
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from src.config import BASELINE_PATH, BASELINE_SUMMARY_PATH, FIGURES
from src.plots import BaselinePlotter, FigureStore, PlotStyle
from src.schema import LIFT_COLS, Col

style = PlotStyle()
style.apply()
plotter, store = BaselinePlotter(style), FigureStore(FIGURES)
weekly = pd.read_parquet(BASELINE_PATH)
summary = pd.read_parquet(BASELINE_SUMMARY_PATH)

## What drives volume, by category
Share of units that is baseline, and the share each driver adds on top. Negative pantry lift means a net post-promotion dip.

In [ ]:
totals = weekly.groupby(Col.CATEGORY)[[Col.UNITS, Col.BASELINE, *LIFT_COLS.values()]].sum()
(totals.div(totals[Col.UNITS], axis=0) * 100).round(1).drop(columns=Col.UNITS)

In [ ]:
store.save(plotter.category_due_to(weekly), "03_due_to_by_category")

## Weekly decomposition, top products by incremental volume
The stacked areas are baseline plus positive lifts; the black line is actual units.

In [ ]:
top = weekly.groupby(Col.UPC)[Col.INCREMENTAL].sum().nlargest(3).index.tolist()
store.save(plotter.weekly(weekly, top), "03_baseline_weekly")

## Bad-baseline corrections
How many weeks each RGMx rule flagged, and how the final baseline was set.

In [ ]:
rules = [c for c in weekly.columns if c.startswith("rule_")]
weekly[rules].sum()

In [ ]:
weekly[Col.BASELINE_FIX].value_counts()

## Product summary
Baseline share of volume and the average lift in promo weeks (units ÷ baseline − 1).

In [ ]:
cols = [Col.DESCRIPTION, Col.PRODUCT_SIZE, Col.CATEGORY, Col.BASELINE_SHARE, Col.PROMO_WEEK_LIFT]
summary[cols].sort_values(Col.PROMO_WEEK_LIFT, ascending=False).head(10).round(2)